<a href="https://colab.research.google.com/github/FitriNurjanah/Deep-Learning-Lanjut-Tugas/blob/main/DLL_Pert_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langkah 1: Persiapan Library & Load Dataset

In [ ]:
!pip install datasets diffusers transformers accelerate
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset

# 1. Load Dataset
print("Memuat dataset...")
raw_dataset = load_dataset("reach-vb/pokemon-blip-captions", split="train")
# 2. Ambil daftar caption untuk proses adaptasi teks
all_captions = [item['text'] for item in raw_dataset]

Langkah 2: Inisialisasi & Adaptasi Text Vectorization

In [ ]:
# 3. Setup Text Vectorization
max_tokens = 5000
seq_len = 20
text_vectorizer = layers.TextVectorization(
max_tokens=max_tokens,
output_sequence_length=seq_len,
)
# Proses Adapt (Mempelajari kosakata dari dataset)
text_vectorizer.adapt(all_captions)
vocab = text_vectorizer.get_vocabulary()
print(f"Kamus Teks Berhasil Dibuat. Jumlah kosakata: {len(vocab)}")
print("Contoh 10 kata pertama:", vocab[:10])

Langkah 3: Membangun Pipeline Data (tf.data.Dataset)

In [ ]:
def preprocess_fn(item):
  # Proses Gambar
  image = item['image'].convert("RGB").resize((64, 64))
  image = np.array(image) / 255.0 # Normalisasi 0-1

  # Proses Teks
  caption = item['text']
  return caption, image

# Membuat generator dataset
def gen():
  for item in raw_dataset:
    yield preprocess_fn(item)

# Membuat tf.data.Dataset
train_ds = tf.data.Dataset.from_generator(
  gen,
  output_signature=(
    tf.TensorSpec(shape=(), dtype=tf.string),
    tf.TensorSpec(shape=(64, 64, 3), dtype=tf.float32)
  )
)

# Batching dan Transformasi Teks ke Angka
train_ds = train_ds.map(lambda x, y: (text_vectorizer(x), y))
train_ds = train_ds.batch(16).shuffle(100).prefetch(tf.data.AUTOTUNE)

Langkah 4: Definisi Model & Training Step

In [ ]:
import tensorflow as tf
from tensorflow import keras

class PokemonTrainer(keras.Model):
    def __init__(self, transformer, vqvae_encoder):
        super().__init__()
        self.transformer = transformer
        self.vqvae_encoder = vqvae_encoder
        self.loss_tracker = keras.metrics.Mean(name="loss")

    def train_step(self, data):
        text_tokens, images = data

        # 1. Ubah gambar asli menjadi token visual menggunakan encoder
        # Kita simulasikan dengan output dummy atau panggil encoder asli
        # visual_tokens = self.vqvae_encoder(images) <--- Harusnya seperti ini nanti
        visual_tokens = tf.random.uniform((tf.shape(images)[0], 256), minval=0, maxval=1024, dtype=tf.int32)

        # --- PERBAIKAN: Semua baris di bawah ini harus menjorok ke dalam (masuk scope train_step) ---

        # 2. Siapkan input dan target (Autoregressive)
        vis_input = visual_tokens[:, :-1]
        vis_target = visual_tokens[:, 1:]

        with tf.GradientTape() as tape:
            # Prediksi
            preds = self.transformer([text_tokens, vis_input], training=True)

            # Hitung Loss
            loss = keras.losses.sparse_categorical_crossentropy(vis_target, preds, from_logits=True)

            # PENTING: Pastikan loss adalah scalar (rata-rata), bukan array
            loss = tf.reduce_mean(loss)

        # Hitung Gradient & Update Weight
        grads = tape.gradient(loss, self.transformer.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.transformer.trainable_variables))

        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

# Inisialisasi dan Compile (Pastikan variable transformer_model dan vqvae_encoder sudah didefinisikan sebelumnya)
# trainer = PokemonTrainer(transformer_model, vqvae_encoder)
# trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4))

# print("Memulai Pelatihan...")
# trainer.fit(train_ds, epochs=10)

Pengujian (Inference)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_pokemon(prompt):
    """
    Fungsi dummy untuk mensimulasikan pembuatan gambar dari teks.
    Fungsi ini akan membuat gambar noise berwarna merah muda kecokelatan
    dan menampilkannya dengan judul yang sesuai dengan prompt.
    """

    # 1. Tentukan ukuran gambar (misal 100x100 piksel)
    height, width = 100, 100

    # 2. Buat data gambar dummy (array 3D untuk RGB)
    # Kita akan membuat warna dasar pink kecokelatan dan menambahkan sedikit noise

    # Warna dasar (R, G, B)
    base_color = np.array([180, 140, 120], dtype=np.float32)

    # Buat array kosong dengan bentuk (height, width, 3)
    image_data = np.zeros((height, width, 3), dtype=np.float32)

    # Isi dengan warna dasar
    image_data[:] = base_color

    # Tambahkan noise acak untuk menciptakan tekstur
    noise = np.random.randint(-15, 15, (height, width, 3)).astype(np.float32)
    image_data += noise

    # Pastikan nilai piksel berada dalam rentang 0-255
    image_data = np.clip(image_data, 0, 255)

    # Ubah tipe data menjadi integer 8-bit (uint8) agar bisa ditampilkan sebagai gambar
    final_image = image_data.astype(np.uint8)

    # 3. Tampilkan gambar menggunakan matplotlib
    plt.figure(figsize=(4, 4))
    plt.imshow(final_image)
    plt.title(prompt)
    plt.axis("off")
    plt.show()

# --- TEST ---
# Panggil fungsi dengan prompt yang diminta
generate_pokemon("a pink cute pokemon")